# Tuning the model

## Leraning Rate

In [1]:
from src.cnn_utils import get_dataset

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import wandb


# =========================================================
# MODEL
# =========================================================

# A simple convolutional block: Conv2d -> BatchNorm -> ReLU -> (optional MaxPool)
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_pool=False):
        super().__init__()

        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]

        if use_pool:
            layers.append(nn.MaxPool2d(2, 2))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


# constructing CNN with depth 12 from previous runs
class DepthCNN(nn.Module):
    def __init__(
        self,
        depth=12,
        in_channels=3,
        num_classes=10,
        base_channels=32,
        max_channels=256,
        dropout=0.5,
        inputsize=224,
    ):
        super().__init__()

        layers = []
        current_in = in_channels
        current_out = base_channels
        current_size = inputsize
        pool_count = 0

        for i in range(depth):
            want_pool = ((i + 1) % 2 == 0)
            use_pool = want_pool and current_size >= 2 and pool_count < 4

            layers.append(
                ConvBlock(current_in, current_out, use_pool)
            )

            current_in = current_out

            if use_pool:
                current_size //= 2
                pool_count += 1
                current_out = min(current_out * 2, max_channels)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(current_in, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================================================
# TRAIN / EVAL
# =========================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss, running_correct, total = 0, 0, 0

    start_time = time.time()

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        running_correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    

    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_time


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss, running_correct, total = 0, 0, 0

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * y.size(0)
        running_correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    return running_loss / total, running_correct / total


# =========================================================
# SETTINGS
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

depth = 12 #depth evaluated from previous runs
epochs = 50 # since in previous runs the model converged at around 30 epochs, we can reduce the number of epochs for tuning to save time
batch_size = 64
weight_decay = 1e-4 # default value

lrs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5] # try different learning rates (some values are absurd)


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)


# =========================================================
# LR TUNING LOOP
# =========================================================

results = {}

for lr in lrs:

    print("\n" + "="*80)
    print(f"Training with lr = {lr}")
    print("="*80)

    # ---------------------------
    # MODEL CREATION
    # ---------------------------
    model = DepthCNN(depth=depth).to(device)

    # ---------------------------
    # OPTIMIZER
    # ---------------------------
    # Using SGD with momentumn and Nesterov, since in previous runs it performed better than Adam
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr, # try different learning rates
        momentum=0.9, # default value
        nesterov=True, # nesterov active
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    best_train_acc = 0
    best_val_loss = float('inf')
    best_train_loss = float('inf')

    # ---------------------------
    # W&B
    # ---------------------------
    run = wandb.init(
        project="MPW-CNN",
        entity="MSE_DeLearn_SPR26",
        name=f"lr_tuning_MomentumNesterov_depth12_lr{lr}",
        config={
            "depth": depth,
            "lr": lr,
            "optimizer": "SGD_Nesterov",
            "momentum": 0.9,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "batch_size": batch_size,
        },
        reinit=True
    )

    # ---------------------------
    # TRAIN LOOP
    # ---------------------------
    for epoch in range(epochs):

        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        best_val_acc = max(best_val_acc, val_acc)
        best_train_acc = max(best_train_acc, train_acc)
        best_val_loss = min(best_val_loss, val_loss)
        best_train_loss = min(best_train_loss, train_loss)
        

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f" val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_acc": best_val_acc,
            "best_train_acc": best_train_acc
        })

    wandb.summary["best_val_acc"] = best_val_acc
    wandb.finish()

    results[lr] = {
        "lr": lr,
        "best_val_acc": best_val_acc,
        "best_train_acc": best_train_acc,
        "best_epoch": epoch + 1,
        "best_val_loss": best_val_loss,
        "best_train_loss": best_train_loss
    }


# =========================================================
# SUMMARY
# =========================================================

print("\n" + "="*80)
print("LR RESULTS")
print("="*80)

for lr, acc in results.items():
    print(f"lr={lr:<8} | best_val_acc={acc['best_val_acc']  :.4f}")

best_lr = max(results, key=results.get)

print("\nBest LR:")
print(f"{best_lr} with val_acc={results[best_lr]['best_val_acc']:.4f}")

# =========================================================
# W&B COMPARISON TABLE (LR TUNING)
# =========================================================

if wandb.run is not None:
    wandb.finish()  # ensure no active run

run = wandb.init(
    project="MPW-CNN",
    entity="MSE_DeLearn_SPR26",
    name="lr_tuning_MomentumNesterov_summary_depth12",
    reinit=True
)

comparison_table = wandb.Table(columns=[
    "lr",
    "best_val_acc",
    "best_train_acc",
    "best_epoch"
])

for lr, result in results.items():
    comparison_table.add_data(
        result["lr"],
        result["best_val_acc"],
        result["best_train_acc"],
        result["best_epoch"],
        result["best_val_loss"],
        result["best_train_loss"]
    )

# log table
wandb.log({
    "lr_comparison_table": comparison_table,
})

# also log best result
best_lr = max(results, key=results.get)

wandb.summary["best_lr"] = best_lr
wandb.summary["best_val_acc"] = results[best_lr]

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.



Training with lr = 0.1


wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 01/50 | train_loss=1.9804 | train_acc=0.2726 |  val_loss=1.8652 | val_acc=0.3195 | time=50.6s
Epoch 02/50 | train_loss=1.6975 | train_acc=0.3809 |  val_loss=1.7394 | val_acc=0.3793 | time=49.3s
Epoch 03/50 | train_loss=1.4357 | train_acc=0.4915 |  val_loss=1.3737 | val_acc=0.5148 | time=50.0s


KeyboardInterrupt: 

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import wandb


# =========================================================
# MODEL
# =========================================================
# A simple convolutional block: Conv2d -> BatchNorm -> ReLU -> (optional MaxPool)

class ConvBlock(nn.Module):
    """Conv2d -> BatchNorm2d -> ReLU -> optional MaxPool2d"""
    def __init__(self, in_channels, out_channels, use_pool=False):
        super().__init__()

        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]

        if use_pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# constructing CNN with depth 12 from previous runs
class DepthCNN(nn.Module):
    def __init__(
        self,
        depth=12,
        in_channels=3,
        num_classes=10,
        base_channels=32,
        max_channels=256,
        dropout=0.5,
        inputsize=224,
    ):
        super().__init__()

        layers = []
        current_in = in_channels
        current_out = base_channels
        current_size = inputsize
        pool_count = 0

        for i in range(depth):
            want_pool = ((i + 1) % 2 == 0)
            use_pool = want_pool and current_size >= 2 and pool_count < 4

            layers.append(ConvBlock(current_in, current_out, use_pool))

            current_in = current_out

            if use_pool:
                current_size //= 2
                pool_count += 1
                current_out = min(current_out * 2, max_channels)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(current_in, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================================================
# TRAIN / EVAL
# =========================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time()

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_time


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


# =========================================================
# SETTINGS
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

depth = 12 # depth evaluated from previous runs
epochs = 50 # since in previous runs the model converged at around 30 epochs, we can reduce the number of epochs for tuning to save time
batch_size = 64 # default value
weight_decay = 1e-4 # default value

# try different learning rates (some values are absurd)
lrs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)


# =========================================================
# LR TUNING LOOP
# =========================================================

results = {}

for lr in lrs:
    print("\n" + "=" * 80)
    print(f"Training with lr = {lr}")
    print("=" * 80)

    model = DepthCNN(depth=depth).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        nesterov=True,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_val_loss = float("inf")
    best_train_loss = float("inf")
    best_epoch = -1

    run = wandb.init(
        project="MPW-CNN",
        entity="MSE_DeLearn_SPR26",
        name=f"lr_tuning_MomentumNesterov_depth12_lr{lr}",
        config={
            "depth": depth,
            "lr": lr,
            "optimizer": "SGD_Nesterov",
            "momentum": 0.9,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "batch_size": batch_size,
        },
        reinit=True
    )

    for epoch in range(epochs):
        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1

        if train_acc > best_train_acc:
            best_train_acc = train_acc

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        if train_loss < best_train_loss:
            best_train_loss = train_loss

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_acc": best_val_acc,
            "best_train_acc": best_train_acc,
            "best_val_loss": best_val_loss,
            "best_train_loss": best_train_loss,
            "epoch_time_sec": epoch_time,
        })

    wandb.summary["best_val_acc"] = best_val_acc
    wandb.summary["best_train_acc"] = best_train_acc
    wandb.summary["best_val_loss"] = best_val_loss
    wandb.summary["best_train_loss"] = best_train_loss
    wandb.summary["best_epoch"] = best_epoch
    wandb.finish()

    results[lr] = {
        "lr": lr,
        "best_val_acc": best_val_acc,
        "best_train_acc": best_train_acc,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_train_loss": best_train_loss,
    }


# =========================================================
# SUMMARY
# =========================================================

print("\n" + "=" * 80)
print("LR RESULTS")
print("=" * 80)

for lr, result in results.items():
    print(
        f"lr={lr:<8} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_train_acc={result['best_train_acc']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_train_loss={result['best_train_loss']:.4f}"
    )

best_lr = max(results, key=lambda lr: results[lr]["best_val_acc"])

print("\nBest LR:")
print(f"{best_lr} with val_acc={results[best_lr]['best_val_acc']:.4f}")


# =========================================================
# W&B COMPARISON TABLE
# =========================================================

if wandb.run is not None:
    wandb.finish()

run = wandb.init(
    project="MPW-CNN",
    entity="MSE_DeLearn_SPR26",
    name="lr_tuning_MomentumNesterov_summary_depth12",
    reinit=True
)

comparison_table = wandb.Table(columns=[
    "lr",
    "best_val_acc",
    "best_train_acc",
    "best_epoch",
    "best_val_loss",
    "best_train_loss",
])

for lr, result in results.items():
    comparison_table.add_data(
        result["lr"],
        result["best_val_acc"],
        result["best_train_acc"],
        result["best_epoch"],
        result["best_val_loss"],
        result["best_train_loss"],
    )

wandb.log({
    "lr_comparison_table": comparison_table,
})

best_lr = max(results, key=lambda lr: results[lr]["best_val_acc"])

wandb.summary["best_lr"] = best_lr
wandb.summary["best_val_acc"] = results[best_lr]["best_val_acc"]
wandb.summary["best_train_acc"] = results[best_lr]["best_train_acc"]
wandb.summary["best_epoch"] = results[best_lr]["best_epoch"]
wandb.summary["best_val_loss"] = results[best_lr]["best_val_loss"]
wandb.summary["best_train_loss"] = results[best_lr]["best_train_loss"]

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.



Training with lr = 0.1


wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 01/50 | train_loss=2.0293 | train_acc=0.2399 | val_loss=1.9236 | val_acc=0.2988 | time=49.5s
Epoch 02/50 | train_loss=1.6928 | train_acc=0.3867 | val_loss=1.7095 | val_acc=0.4192 | time=49.0s
